# SONDA: processamento e visualizacao individual

Este notebook carrega as bases finais SONDA-MD e SONDA-SD, aplica os periodos explicitamente definidos para cada uma e gera as figuras historicas individuais.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180

def encontrar_pasta_2026(inicio=Path.cwd()):
    inicio = inicio.resolve()
    for pasta in (inicio, *inicio.parents):
        candidata = pasta / '2026'
        if candidata.exists():
            return candidata
    raise FileNotFoundError('Nao foi encontrada a pasta 2026 a partir do diretorio atual.')

PASTA_2026 = encontrar_pasta_2026()
PASTA_SAIDA = PASTA_2026 / 'visualizacoes_comparativas_bases'
PASTA_FONTE = PASTA_SAIDA / 'figuras_por_fonte'
PASTA_FONTE.mkdir(parents=True, exist_ok=True)

FREQUENCIA_VISUAL = 'D'

ROTULOS = {
    'radiacao_onda_curta': 'Radiacao de onda curta (W/m2)',
    'temperatura': 'Temperatura (C)',
    'umidade_relativa': 'Umidade relativa (%)',
}

def recortar_periodo(dados, inicio, fim):
    return dados.loc[
        (dados['data_hora'] >= inicio) & (dados['data_hora'] <= fim)
    ].copy().reset_index(drop=True)

def serie_visual(dados, coluna):
    return dados.set_index('data_hora')[coluna].resample(FREQUENCIA_VISUAL).mean()

def plotar_evolucao(dados, nome, colunas, arquivo, ncol=1):
    colunas = [coluna for coluna in colunas if coluna in dados.columns]
    nlin = int(np.ceil(len(colunas) / ncol))
    fig, eixos = plt.subplots(nlin, ncol, figsize=(14, 3.6 * nlin), sharex=True, squeeze=False)
    for eixo, coluna in zip(eixos.flat, colunas):
        serie_visual(dados, coluna).plot(ax=eixo, color='#1f5a85', linewidth=0.8)
        eixo.set_title(ROTULOS.get(coluna, coluna))
        eixo.set_xlabel('')
        eixo.set_ylabel('')
    for eixo in eixos.flat[len(colunas):]:
        eixo.remove()
    inicio = dados['data_hora'].min().strftime('%d/%m/%Y')
    fim = dados['data_hora'].max().strftime('%d/%m/%Y')
    fig.suptitle(f'Evolucao temporal - {nome} ({inicio} a {fim}; media diaria)', y=1.01)
    fig.tight_layout()
    fig.savefig(arquivo, bbox_inches='tight')
    plt.show()
    plt.close(fig)

PASTA_SONDA_TRATADA = PASTA_2026 / 'dados_tratados_sonda'
INICIO_MD = pd.Timestamp('2024-01-05 00:00:00')
FIM_MD = pd.Timestamp('2026-01-05 23:00:00')
INICIO_SD = pd.Timestamp('2024-02-07 16:00:00')
FIM_SD = pd.Timestamp('2026-06-30 23:00:00')

def ler_sonda(nome_arquivo, colunas, inicio, fim):
    arquivo = PASTA_SONDA_TRATADA / nome_arquivo
    if not arquivo.exists():
        raise FileNotFoundError(f'Base SONDA tratada nao encontrada: {arquivo}. Execute primeiro analise_dados_sonda.ipynb.')
    dados = pd.read_csv(arquivo, parse_dates=['data_hora'])
    for coluna in colunas:
        dados[coluna] = pd.to_numeric(dados[coluna], errors='coerce')
    dados = dados.dropna(subset=['data_hora']).sort_values('data_hora').drop_duplicates('data_hora')
    return recortar_periodo(dados, inicio, fim)

sonda_md = ler_sonda('sonda_md_horario_padronizado.csv', ['temperatura', 'umidade_relativa'], INICIO_MD, FIM_MD)
sonda_sd = ler_sonda('sonda_sd_depois_lacuna.csv', ['radiacao_onda_curta'], INICIO_SD, FIM_SD)

print(f'SONDA-MD: {len(sonda_md)} registros | {sonda_md.data_hora.min()} a {sonda_md.data_hora.max()}')
display(sonda_md.describe(include='all'))
print(f'SONDA-SD: {len(sonda_sd)} registros | {sonda_sd.data_hora.min()} a {sonda_sd.data_hora.max()}')
display(sonda_sd.describe(include='all'))


In [ ]:
from matplotlib.patches import Patch

def plotar_temperatura_sonda_md(dados, arquivo):
    diario = dados.set_index('data_hora')['temperatura'].resample(FREQUENCIA_VISUAL).mean()
    fig, eixo = plt.subplots(figsize=(14, 4.2))
    diario.plot(ax=eixo, color='#1f5a85', linewidth=0.8)
    eixo.set_title('Temperatura (C)')
    eixo.set_xlabel('Data')
    eixo.set_ylabel('')
    fig.suptitle('Evolucao temporal - SONDA-MD: temperatura (05/01/2024 a 05/01/2026; media diaria)', y=1.01)
    fig.tight_layout()
    fig.savefig(arquivo, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def plotar_umidade_sonda_md(dados, arquivo):
    diario = dados.set_index('data_hora')['umidade_relativa'].resample(FREQUENCIA_VISUAL).mean()
    cobertura = dados.set_index('data_hora')['umidade_relativa'].resample(FREQUENCIA_VISUAL).count()
    dias_parciais = cobertura[cobertura < 24].index
    fig, eixo = plt.subplots(figsize=(14, 4.2))
    diario.plot(ax=eixo, color='#1f5a85', linewidth=0.8)
    for dia in dias_parciais:
        eixo.axvspan(dia, dia + pd.Timedelta(days=1), color='#e6a700', alpha=0.22, linewidth=0)
    eixo.set_title('Umidade relativa (%)')
    eixo.set_xlabel('Data')
    eixo.set_ylabel('')
    eixo.legend(handles=[Patch(facecolor='#e6a700', alpha=0.22, label='Dia com cobertura parcial de umidade')], loc='best', fontsize=8)
    fig.suptitle('Evolucao temporal - SONDA-MD: umidade relativa (05/01/2024 a 05/01/2026; media diaria)', y=1.01)
    fig.tight_layout()
    fig.savefig(arquivo, bbox_inches='tight')
    plt.show()
    plt.close(fig)


In [ ]:
plotar_temperatura_sonda_md(sonda_md, PASTA_FONTE / 'sonda_md_temperatura_evolucao.png')
plotar_umidade_sonda_md(sonda_md, PASTA_FONTE / 'sonda_md_umidade_evolucao.png')
plotar_evolucao(sonda_sd, 'SONDA-SD - apos a lacuna', ['radiacao_onda_curta'], PASTA_FONTE / 'sonda_sd_apos_lacuna_evolucao.png')
